<a href="https://colab.research.google.com/github/muppadivignesh-del/Machine-Learning_Lab_Experiments/blob/main/Exp_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

In [ ]:
path = "/content/drive/MyDrive/Colab/placement_predict_50k_adjusted.csv"

df = pd.read_csv(path)

print("Dataset shape:", df.shape)
df.head()

In [ ]:
NUMERIC_COLS = [
    "CGPA", "AttendancePercent", "Internships", "Projects",
    "Workshops", "Certifications", "Publications",
    "AptitudeTestScore", "SoftSkillsRating",
    "CodingTestScore", "MockInterviewScore"
]

CATEGORICAL_COLS = [
    "Gender", "City", "CollegeTier", "Stream",
    "Specialisation", "Hostel", "HistoryOfBacklogs",
    "ExtraCurricular"
]

In [ ]:
imputer = SimpleImputer(strategy="median")

num_df = pd.DataFrame(
    imputer.fit_transform(df[NUMERIC_COLS]),
    columns=NUMERIC_COLS
)

cat_df = pd.get_dummies(df[CATEGORICAL_COLS], drop_first=True)

feature_df = pd.concat([num_df, cat_df], axis=1)

X = StandardScaler().fit_transform(feature_df)

print("Clustering data:", X.shape)

In [ ]:
scores = {}

for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X)
    scores[k] = silhouette_score(X, labels)

print(scores)

best_k = max(scores, key=scores.get)
print("Best K:", best_k)

In [ ]:
scores = {}

for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X)
    scores[k] = silhouette_score(X, labels)

print(scores)

best_k = max(scores, key=scores.get)
print("Best K:", best_k)

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

plt.figure(figsize=(7,5))
plt.scatter(X_pca[:,0], X_pca[:,1],
            c=df["KMeansCluster"], s=8)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Student K-Means Clusters")
plt.show()

In [ ]:
sample = np.random.RandomState(42).choice(len(X), 5000, replace=False)
X_sample = X[sample]

hc = AgglomerativeClustering(n_clusters=best_k)
hc_labels = hc.fit_predict(X_sample)

db = DBSCAN(eps=4.0, min_samples=15)
db_labels = db.fit_predict(X_sample)

print("Hierarchical clusters:", len(set(hc_labels)))
print("DBSCAN clusters:", len(set(db_labels)) - (1 if -1 in db_labels else 0))

In [ ]:
output = "/content/drive/MyDrive/Colab/placement_predict_with_clusters.csv"

df.to_csv(output, index=False)

print("Saved successfully!")
print(output)

In [ ]:
profile = df.groupby("KMeansCluster").agg(
    Students=("PlacementStatus", "size"),
    Avg_CGPA=("CGPA", "mean"),
    Avg_Attendance=("AttendancePercent", "mean"),
    Avg_Coding=("CodingTestScore", "mean"),
    Avg_Interview=("MockInterviewScore", "mean"),
    Avg_Internships=("Internships", "mean"),
    Placement_Rate=("PlacementStatus", "mean")
).round(2)

profile

In [ ]:
profile.to_csv(
    "/content/drive/MyDrive/Colab/cluster_profiles.csv"
)

print("Cluster profile saved to Google Drive!")